# 🚀 LLM Playground — Full Pipeline on Colab T4

In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ Runtime → Change runtime type → T4 GPU")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.6 GB


## Setup

In [2]:
import os

REPO_URL = "https://github.com/mtue04/llm-playground"
REPO_DIR = "/content/llm-playground"

if not os.path.exists(REPO_DIR):
    if os.path.exists("/content/llm-playground.zip"):
        print("📦 Found llm-playground.zip. Unzipping...")
        !unzip -q /content/llm-playground.zip -d /content/
    else:
        print("🌐 Cloning repository from GitHub...")
        !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}. Checking for updates...")
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}

Repo already cloned at /content/llm-playground. Checking for updates...
/content/llm-playground
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 11 (delta 9), reused 11 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 1.00 KiB | 342.00 KiB/s, done.
From https://github.com/mtue04/llm-playground
   a0fa31a..4df213d  main       -> origin/main
Updating a0fa31a..4df213d
Fast-forward
 evaluation/multiple_choice_eval.py | 2 +-
 evaluation/perplexity.py           | 2 +-
 inference/generate.py              | 2 +-
 training/rl/reward_model.py        | 2 +-
 training/trainer_utils.py          | 2 +-
 5 files changed, 5 insertions(+), 5 deletions(-)
/content/llm-playground


In [3]:
import os

def print_tree(startpath, max_depth=3):
    for root, dirs, files in os.walk(startpath):
        # Bỏ qua thư mục ẩn và cache
        dirs[:] = [d for d in dirs if not d.startswith('.') and d != '__pycache__']
        
        level = root.replace(startpath, '').count(os.sep)
        if level >= max_depth:
            continue
            
        indent = ' ' * 4 * level
        print(f'{indent}{os.path.basename(root) or startpath}/')
        
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            # Chỉ hiện các file không phải file ẩn/file cache
            if not f.startswith('.'):
                print(f'{subindent}{f}')

print_tree('.')


./
    README.md
    requirements.txt
    tokenizer/
        train_tokenizer.py
        __init__.py
        vocab/
            tokenizer.json
    checkpoints/
        pretrain_step3500.pt
        pretrain_step2000.pt
        pretrain_step4000.pt
        pretrain_step3000.pt
        pretrain_step4500.pt
        pretrain_final.pt
        pretrain_step1000.pt
        pretrain_step1500.pt
        pretrain_step500.pt
        pretrain_step2500.pt
    evaluation/
        human_eval_ui.py
        multiple_choice_eval.py
        perplexity.py
        __init__.py
    training/
        trainer_utils.py
        pretrain.py
        sft.py
        __init__.py
        rl/
            reward_model.py
            dpo.py
            __init__.py
    data/
        __init__.py
        raw/
            corpus.txt
        processed/
            corpus_clean.txt
            train.bin
            val.bin
        scripts/
            download.py
            tokenize_corpus.py
            clean.py
            __

In [4]:
!pip install -q -r requirements.txt

In [5]:
import sys
sys.path.insert(0, REPO_DIR)

## Download & Clean Data

In [6]:
TARGET_MB = 100

!python data/scripts/download.py --target_mb {TARGET_MB} --output data/raw/corpus.txt

Resolving data files: 100% 27468/27468 [00:05<00:00, 4663.29it/s]
  ...2.9MB, 1000 documents
  ...6.1MB, 2000 documents
  ...8.8MB, 3000 documents
  ...11.7MB, 4000 documents
  ...14.5MB, 5000 documents
  ...17.4MB, 6000 documents
  ...20.5MB, 7000 documents
  ...23.5MB, 8000 documents
  ...26.4MB, 9000 documents
  ...29.3MB, 10000 documents
  ...32.5MB, 11000 documents
  ...35.4MB, 12000 documents
  ...38.6MB, 13000 documents
  ...41.4MB, 14000 documents
  ...44.4MB, 15000 documents
  ...47.2MB, 16000 documents
  ...50.6MB, 17000 documents
  ...53.7MB, 18000 documents
  ...56.6MB, 19000 documents
  ...59.5MB, 20000 documents
  ...62.4MB, 21000 documents
  ...65.1MB, 22000 documents
  ...68.3MB, 23000 documents
  ...71.2MB, 24000 documents
  ...74.2MB, 25000 documents
  ...77.6MB, 26000 documents
  ...80.6MB, 27000 documents
  ...83.8MB, 28000 documents
  ...86.5MB, 29000 documents
  ...89.5MB, 30000 documents
  ...92.3MB, 31000 documents
  ...95.5MB, 32000 documents
  ...98.6MB, 33000

In [7]:
!python data/scripts/clean.py \
    --input data/raw/corpus.txt \
    --output data/processed/corpus_clean.txt \
    --min_words 20

Total: 33451 | Kept: 33449 | Low quality: 0 | Duplicates: 2


In [8]:
import os
raw_size = os.path.getsize("data/raw/corpus.txt") / (1024*1024)
clean_size = os.path.getsize("data/processed/corpus_clean.txt") / (1024*1024)
print(f"Raw corpus:   {raw_size:.1f} MB")
print(f"Clean corpus: {clean_size:.1f} MB")
print(f"Removed:      {raw_size - clean_size:.1f} MB ({(1 - clean_size/raw_size)*100:.1f}%)")

Raw corpus:   100.1 MB
Clean corpus: 100.1 MB
Removed:      0.0 MB (0.0%)


## Train Tokenizer (BBPE)

In [9]:
VOCAB_SIZE = 8192

!python tokenizer/train_tokenizer.py \
    --input data/processed/corpus_clean.txt \
    --vocab_size {VOCAB_SIZE} \
    --output tokenizer/vocab

[00:00:00] Tokenize words                 ██████████████████ 371420   /   371420[00:00:00] Tokenize words                 ██████████████████ 0        /        0
[00:00:00] Count pairs                    ██████████████████ 371420   /   371420
[00:00:02] Compute merges                 ██████████████████ 7986     /     7986
Tokenizer saved to tokenizer/vocab/tokenizer.json
Vocabulary size: 8192


In [10]:
# Test tokenizer
from tokenizer import LLMTokenizer

tok = LLMTokenizer("tokenizer/vocab/tokenizer.json")
test_text = "Hello, how are you? Machine learning is fascinating!"
ids = tok.encode(test_text, add_bos=True, add_eos=True)
decoded = tok.decode(ids)

print(f"Vocab size: {tok.vocab_size}")
print(f"Special tokens — BOS: {tok.bos_id}, EOS: {tok.eos_id}, PAD: {tok.pad_id}")
print(f"\nOriginal:  {test_text}")
print(f"Token IDs: {ids}")
print(f"Num tokens: {len(ids)}")
print(f"Decoded:   {decoded}")

Vocab size: 8192
Special tokens — BOS: 2, EOS: 3, PAD: 0

Original:  Hello, how are you? Machine learning is fascinating!
Token IDs: [2, 575, 6403, 17, 629, 316, 271, 36, 7901, 424, 2872, 269, 7110, 209, 701, 6, 3]
Num tokens: 17
Decoded:    Hello, how are you? Machine learning is fascinating!


## Tokenize Corpus → Binary Files

In [11]:
!python data/scripts/tokenize_corpus.py \
    --input data/processed/corpus_clean.txt \
    --tokenizer tokenizer/vocab/tokenizer.json \
    --output_dir data/processed \
    --val_ratio 0.02

Tokenizing 33449 documents using fast batch encoding...
Tokenized 27250128 tokens: 26705126 for training, 545002 for validation.
Saved to data/processed/train.bin and data/processed/val.bin.


In [12]:
# Checking binary files
import numpy as np

train_data = np.fromfile("data/processed/train.bin", dtype=np.uint32)
val_data = np.fromfile("data/processed/val.bin", dtype=np.uint32)

print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens:   {len(val_data):,}")
print(f"Total tokens: {len(train_data) + len(val_data):,}")
print(f"\nFirst 20 train tokens: {train_data[:20]}")
print(f"Max token ID: {train_data.max()} (vocab_size = {VOCAB_SIZE})")

Train tokens: 26,705,126
Val tokens:   545,002
Total tokens: 27,250,128

First 20 train tokens: [   2  768   59  716  228 4847  242 2396 2306   31 1419   84  288  309
  275  212 5874  237 3314 1495]
Max token ID: 8191 (vocab_size = 8192)


## Pretrain GPT Model

In [13]:
from model.config import ModelConfig
cfg = ModelConfig.from_yaml("configs/model_small.yaml")
print(f"Model config:")
print(f"  d_model:        {cfg.d_model}")
print(f"  n_layer:        {cfg.n_layer}")
print(f"  n_head:         {cfg.n_head}")
print(f"  head_dim:       {cfg.head_dim}")
print(f"  d_ff:           {cfg.d_ff}")
print(f"  vocab_size:     {cfg.vocab_size}")
print(f"  context_length: {cfg.context_length}")
print(f"  Est. params:    {cfg.num_params() / 1e6:.2f}M")

Model config:
  d_model:        384
  n_layer:        6
  n_head:         6
  head_dim:       64
  d_ff:           1536
  vocab_size:     8192
  context_length: 512
  Est. params:    13.76M


In [14]:
!python training/pretrain.py \
    --model_config configs/model_small.yaml \
    --train_config configs/train_pretrain.yaml

Training on data/processed/train.bin, validating on data/processed/val.bin
Model: 13.77M parameters
step    0 | train_loss 9.1116 | lr 0.00e+00
step   10 | train_loss 8.9420 | lr 1.50e-05
step   20 | train_loss 8.6771 | lr 3.00e-05
step   30 | train_loss 8.4328 | lr 4.50e-05
step   40 | train_loss 8.3322 | lr 6.00e-05
step   50 | train_loss 8.1257 | lr 7.50e-05
step   60 | train_loss 7.9105 | lr 9.00e-05
step   70 | train_loss 7.7658 | lr 1.05e-04
step   80 | train_loss 7.6040 | lr 1.20e-04
step   90 | train_loss 7.3491 | lr 1.35e-04
step  100 | train_loss 7.2728 | lr 1.50e-04
step  110 | train_loss 7.1481 | lr 1.65e-04
step  120 | train_loss 7.1559 | lr 1.80e-04
step  130 | train_loss 7.2360 | lr 1.95e-04
step  140 | train_loss 7.2004 | lr 2.10e-04
step  150 | train_loss 7.0081 | lr 2.25e-04
step  160 | train_loss 7.0026 | lr 2.40e-04
step  170 | train_loss 6.9329 | lr 2.55e-04
step  180 | train_loss 6.7544 | lr 2.70e-04
step  190 | train_loss 6.7447 | lr 2.85e-04
step  200 | train_lo

## Evaluate Pretrained Model

In [15]:
!python evaluation/perplexity.py \
    --checkpoint checkpoints/pretrain_final.pt \
    --model_config configs/model_small.yaml \
    --val_bin data/processed/val.bin \
    --n_batches 50

Val loss: 4.8921
Perplexity: 133.23
(Comparison: random model perplexity ≈ 8192)


In [16]:
# Test generation after pretrain (not SFT, just next-token prediction)
from inference.generate import load_model_for_inference, generate
from tokenizer import LLMTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model_for_inference("checkpoints/pretrain_final.pt", device=device)
tokenizer = LLMTokenizer("tokenizer/vocab/tokenizer.json")

prompts = [
    "The meaning of life is",
    "Machine learning can be used to",
    "In the future, artificial intelligence will",
]

for prompt in prompts:
    _, generated = generate(model, tokenizer, prompt, max_new_tokens=80,
                            temperature=0.8, top_k=50, top_p=0.9, device=device)
    print(f"Prompt: {prompt}")
    print(f"Output: {generated.strip()}")
    print("-" * 60)

Prompt: The meaning of life is
Output: in the world’s endeous in the world. The most interesting, the world is that has been on the world. In addition to its way, the world has its most strongest world in the world, and with the most popular world, it is in the world. The world is the top one of the world’s most-western world in the world. The world is
------------------------------------------------------------
Prompt: Machine learning can be used to
Output: be a part of the way, as well as an artistic and creative. The most importantly we can also be the most exciting and modern history of the world, and that is the best part of the world. We are proud to take care of the world and our future to the world, we are at the first time in the world of the world. We have also seen a variety of
------------------------------------------------------------
Prompt: In the future, artificial intelligence will
Output: offer its own business, and the most of the future. As the first time in the e

## SFT (Supervised Fine-Tuning)

In [17]:
!python training/sft.py \
    --model_config configs/model_small.yaml \
    --train_config configs/train_sft.yaml

Loaded checkpoint from checkpoints/pretrain_final.pt (step 5000)
Loading SFT dataset...
README.md: 100% 5.61k/5.61k [00:00<00:00, 15.3MB/s]
data/train-00000-of-00001.parquet: 100% 10.5M/10.5M [00:01<00:00, 8.66MB/s]
data/test-00000-of-00001.parquet: 100% 571k/571k [00:00<00:00, 668kB/s]
Generating train split: 100% 9500/9500 [00:00<00:00, 120235.75 examples/s]
Generating test split: 100% 500/500 [00:00<00:00, 87519.91 examples/s]
Processed SFT samples: 2000
step    0 | sft_loss 5.0141 | lr 0.00e+00
step   10 | sft_loss 5.2024 | lr 1.00e-05
step   20 | sft_loss 4.7341 | lr 2.00e-05
step   30 | sft_loss 4.9120 | lr 3.00e-05
step   40 | sft_loss 4.5471 | lr 4.00e-05
step   50 | sft_loss 4.5906 | lr 5.00e-05
step   60 | sft_loss 4.5644 | lr 5.00e-05
step   70 | sft_loss 4.4672 | lr 4.99e-05
step   80 | sft_loss 4.4727 | lr 4.98e-05
step   90 | sft_loss 4.4630 | lr 4.97e-05
step  100 | sft_loss 4.3140 | lr 4.95e-05
step  110 | sft_loss 4.4760 | lr 4.93e-05
step  120 | sft_loss 4.4066 | lr 4

In [18]:
# Test generation after SFT
model_sft = load_model_for_inference("checkpoints/sft_final.pt", device=device)

sft_prompts = [
    "### Instruction:\nExplain what machine learning is in simple terms.\n\n### Response:\n",
    "### Instruction:\nWrite a haiku about programming.\n\n### Response:\n",
    "### Instruction:\nList 3 benefits of regular exercise.\n\n### Response:\n",
]

for prompt in sft_prompts:
    _, generated = generate(model_sft, tokenizer, prompt, max_new_tokens=150,
                            temperature=0.7, top_k=50, top_p=0.9, device=device)
    print(f"Prompt: {prompt.strip()}")
    print(f"Response: {generated.strip()}")
    print("=" * 60)

Prompt: ### Instruction:
Explain what machine learning is in simple terms.

### Response:
Response: 1. Eveard is a beautiful and beautiful and powerful way to grow. It is the perfect place to keep you back to the right world.
Prompt: ### Instruction:
Write a haiku about programming.

### Response:
Response: Everyone is a great way to learn.
Prompt: ### Instruction:
List 3 benefits of regular exercise.

### Response:
Response: 1. Average 2-2 time is a variety of ideas that help your body with the highest-story use of a body level.
2. Average A-5. Target and the northern-marer - The work can be used to improve the body.
3. Auburn D-B-F-3 - C-G-E-F-B-R-S-E-2 - This is a great way to create a simple one-time.


## Multiple Choice Evaluation

In [19]:
!python evaluation/multiple_choice_eval.py \
    --checkpoint checkpoints/sft_final.pt \
    --model_config configs/model_small.yaml

Accuracy on 2 sample questions: 50.00%


## DPO (Direct Preference Optimization)

In [20]:
!python training/rl/dpo.py \
    --model_config configs/model_small.yaml \
    --sft_checkpoint checkpoints/sft_final.pt \
    --max_steps 200 \
    --beta 0.1

Loaded checkpoint from checkpoints/sft_final.pt (step 800)
step    0 | dpo_loss 0.6931 | accuracy 0.00%
step   10 | dpo_loss 0.0037 | accuracy 100.00%
step   20 | dpo_loss 0.0006 | accuracy 100.00%
step   30 | dpo_loss 0.0003 | accuracy 100.00%
step   40 | dpo_loss 0.0002 | accuracy 100.00%
step   50 | dpo_loss 0.0002 | accuracy 100.00%
step   60 | dpo_loss 0.0001 | accuracy 100.00%
step   70 | dpo_loss 0.0001 | accuracy 100.00%
step   80 | dpo_loss 0.0001 | accuracy 100.00%
step   90 | dpo_loss 0.0001 | accuracy 100.00%
step  100 | dpo_loss 0.0001 | accuracy 100.00%
step  110 | dpo_loss 0.0001 | accuracy 100.00%
step  120 | dpo_loss 0.0001 | accuracy 100.00%
step  130 | dpo_loss 0.0001 | accuracy 100.00%
step  140 | dpo_loss 0.0001 | accuracy 100.00%
step  150 | dpo_loss 0.0000 | accuracy 100.00%
step  160 | dpo_loss 0.0000 | accuracy 100.00%
step  170 | dpo_loss 0.0000 | accuracy 100.00%
step  180 | dpo_loss 0.0000 | accuracy 100.00%
step  190 | dpo_loss 0.0000 | accuracy 100.00%
Sav

## Save to Google Drive

In [21]:
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    import shutil
    DRIVE_DIR = "/content/drive/MyDrive/llm-playground-checkpoints"
    os.makedirs(DRIVE_DIR, exist_ok=True)

    # Copy checkpoints
    for f in os.listdir("checkpoints"):
        src = os.path.join("checkpoints", f)
        dst = os.path.join(DRIVE_DIR, f)
        shutil.copy2(src, dst)
        print(f"Saved: {dst}")

    # Copy tokenizer
    tok_dst = os.path.join(DRIVE_DIR, "tokenizer.json")
    shutil.copy2("tokenizer/vocab/tokenizer.json", tok_dst)
    print(f"Saved tokenizer: {tok_dst}")

    print(f"\n✅ All checkpoints saved to Google Drive: {DRIVE_DIR}")
else:
    print("Skipping Drive save. Set SAVE_TO_DRIVE = True to enable.")

Mounted at /content/drive
Saved: /content/drive/MyDrive/llm-playground-checkpoints/sft_final.pt
Saved: /content/drive/MyDrive/llm-playground-checkpoints/pretrain_step3500.pt
Saved: /content/drive/MyDrive/llm-playground-checkpoints/pretrain_step2000.pt
Saved: /content/drive/MyDrive/llm-playground-checkpoints/pretrain_step4000.pt
Saved: /content/drive/MyDrive/llm-playground-checkpoints/pretrain_step3000.pt
Saved: /content/drive/MyDrive/llm-playground-checkpoints/dpo_final.pt
Saved: /content/drive/MyDrive/llm-playground-checkpoints/pretrain_step4500.pt
Saved: /content/drive/MyDrive/llm-playground-checkpoints/pretrain_final.pt
Saved: /content/drive/MyDrive/llm-playground-checkpoints/sft_step400.pt
Saved: /content/drive/MyDrive/llm-playground-checkpoints/pretrain_step1000.pt
Saved: /content/drive/MyDrive/llm-playground-checkpoints/pretrain_step1500.pt
Saved: /content/drive/MyDrive/llm-playground-checkpoints/pretrain_step500.pt
Saved: /content/drive/MyDrive/llm-playground-checkpoints/pretrai